In [4]:
import requests
import json
from pprint import pprint
import os
from dotenv import load_dotenv
from pprint import pprint

# Load environment variables from repo root
load_dotenv("../.env")

api_key = os.getenv("SURF_API_KEY")
willma_base_url = os.getenv("SURF_API_BASE", "https://willma.surf.nl/api/v0")

if not api_key:
    raise ValueError("SURF_API_KEY is not set in .env")

headers = {"X-API-KEY": api_key, "Content-Type": "application/json"}

models = requests.get(f"{willma_base_url}/sequences", headers=headers, timeout=30).json()
text_models = [m["name"] for m in models if m.get("sequence_type") == "text"]

print("Available text models:", text_models)
model_name = "default-text-large" if "default-text-large" in text_models else (text_models[0] if text_models else None)
if not model_name:
    raise RuntimeError("No text models returned by /sequences")
pprint(text_models)


Available text models: ['default-text-large', 'openai/gpt-oss-120b', 'Sehyo/Qwen3.5-122B-A10B-NVFP4', 'mistralai/Devstral-Small-2-24B-Instruct-2512', 'RedHatAI/gemma-4-31B-it-NVFP4', 'Qwen/Qwen3.6-35B-A3B-FP8', 'Qwen/Qwen3.6-27B-FP8', 'Qwen/Qwen2.5-Coder-32B-Instruct-AWQ', 'Qwen/Qwen2.5-VL-32B-Instruct-AWQ']
['default-text-large',
 'openai/gpt-oss-120b',
 'Sehyo/Qwen3.5-122B-A10B-NVFP4',
 'mistralai/Devstral-Small-2-24B-Instruct-2512',
 'RedHatAI/gemma-4-31B-it-NVFP4',
 'Qwen/Qwen3.6-35B-A3B-FP8',
 'Qwen/Qwen3.6-27B-FP8',
 'Qwen/Qwen2.5-Coder-32B-Instruct-AWQ',
 'Qwen/Qwen2.5-VL-32B-Instruct-AWQ']


In [ ]:
response = requests.post(
    f"{willma_base_url}/chat/completions",
    data=json.dumps({
        "model": "Qwen/Qwen3.6-35B-A3B-FP8",
        "messages": [{"role": "user", "content": "Hoi!"}],
    }),
    headers=headers,
    timeout=4600,
).json()


In [3]:
response

{'id': 'chatcmpl-trace_7afa06e1-3f5d-4556-b3ce-ac933b94b677',
 'created': 1784106233,
 'model': 'mistralai/Devstral-Small-2-24B-Instruct-2512',
 'choices': [{'index': 0,
   'message': {'role': 'assistant', 'content': 'Hoi! Hoe kan ik je helpen? 😊'},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 6, 'completion_tokens': 15, 'total_tokens': 21},
 'surf': {'gpu_power': 0.0, 'compute_time_s': 1.5391}}

The selected model is discovered from `/sequences` and stored in `model_name`.

If you want a fixed model, set `model_name` manually after running the first cell.


## Structured output probe

Tests whether each SURF text model supports structured output via LangChain `with_structured_output()` — the same mechanism the metadata agent uses during synthesis (`Player._synthesize_structured`).

Set `probe_models = text_models` to test every model, or `probe_models = [model_name]` for a single model.

In [ ]:
import sys

sys.path.insert(0, "..")

from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.prompts import ChatPromptTemplate
from src.config import create_llm

class MinimalMetadata(BaseModel):
    """Tiny stand-in for production metadata schemas."""
    title: str = Field(description="Dataset title")
    description: str = Field(description="One-sentence dataset description")
    format: Optional[str] = Field(default=None, description="File or data format")

SAMPLE_TASK = "Extract metadata for a CSV of bird observations in the Netherlands."
SAMPLE_CONTEXT = """
Analyst notes:
- File: observations.csv, 12 columns, ~50k rows
- Columns include species, date, latitude, longitude
- Coverage: Netherlands, 2018–2022
""".strip()

STRUCTURED_METHODS: list[str | None] = [None, "function_calling", "json_schema", "json_mode"]

# Test one model (fast) or all models from /sequences
# probe_models = [model_name]
probe_models = text_models


def probe_structured_output(model: str) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You synthesize analyst notes into the required JSON schema. Use concrete values from the notes."),
        ("human", "Task: {task}\n\nAnalyst notes:\n{context}\n\nFill every required field."),
    ])

    for method in STRUCTURED_METHODS:
        tag = method or "(default)"
        try:
            llm = create_llm(temperature=0.0, model_name=model)
            kwargs = {} if method is None else {"method": method}
            structured_llm = llm.with_structured_output(MinimalMetadata, **kwargs)
            result = (prompt | structured_llm).invoke({
                "task": SAMPLE_TASK,
                "context": SAMPLE_CONTEXT,
            })
            return {
                "model": model,
                "ok": True,
                "method": tag,
                "result": result.model_dump(),
            }
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"

    return {
        "model": model,
        "ok": False,
        "method": None,
        "error": last_error,
    }


results = []
for model in probe_models:
    print(f"\n{'=' * 60}\nProbing structured output: {model}\n{'=' * 60}")
    outcome = probe_structured_output(model)
    results.append(outcome)
    if outcome["ok"]:
        print(f"OK with method={outcome['method']}")
        pprint(outcome["result"])
    else:
        print("FAILED — no method worked")
        print(outcome["error"])

supported = [r["model"] for r in results if r["ok"]]
unsupported = [r["model"] for r in results if not r["ok"]]
print(f"\nSummary: {len(supported)}/{len(results)} model(s) support structured output")
if supported:
    print("  supported:", supported)
if unsupported:
    print("  unsupported:", unsupported)